# Step 3: Validation of classification models

This notebook provides instructions for train/test validation of Random Forest models on k‑mer features (MetaFX) and taxonomic profiles (Kraken2), as well as external validation on an arthritis cohort.

All scripts are located in `pipeline/03_validation/`.

## Prerequisites
- Completed Step 2 (MetaFX feature tables, trained model, contigs).
- Completed Step 1 (Kraken species table `kraken_species_reads.csv`).
- Independent arthritis dataset (FASTQ files and metadata).

## 3.1 Train/test split and validation with MetaFX (k‑mer features)

### 3.1.1 Split data

In [ ]:
cd pipeline/03_validation
python split_train_test.py

### 3.1.2 Copy kmers for training set

In [ ]:
sbatch copy_kmers_train.sbatch

### 3.1.3 Run metafx unique on training set

In [ ]:
sbatch run_unique_train.sbatch

### 3.1.4 Compute features for test set

In [ ]:
sbatch calc_features_test.sbatch

### 3.1.5 Preprocess feature tables (filter low‑prevalence k‑mers, normalise)

In [ ]:
python preprocess_metafx.py

### 3.1.6 Train model and predict

In [ ]:
bash run_fit_predict.sh

### 3.1.7 Evaluate predictions

In [ ]:
python evaluate_predictions.py

### 3.1.8 Extract top‑20 contigs from the trained model

In [ ]:
python extract_top20_contigs_preproc.py

Result: top20_contigs_metafx_preproc.fasta (to be annotated with BLAST).

## 3.2 Validation with Kraken2 taxonomic profiles

This script:

Loads Kraken species read counts

Applies 5% prevalence filter

Trains Random Forest on the predefined training set

Evaluates on test set

Saves confusion matrix to ../../images/Step3_kraken_confusion_matrix.png

In [ ]:
python kraken_train_test.py

![Confusion matrix](../images/Step3_kraken_confusion_matrix.png)

Top-10 most important taxa:

||taxon|  importance|
|----|------------------------------------------|------------|
|2856|   Clostridium saccharoperbutylacetonicum |   0.013596|
|2844|                      Clostridium gelidum |   0.011232|
|5746|            Macellibacteroides fermentans|    0.010464|
|6302|    Methylophaga nitratireducenticrescens|    0.009682|
|8184|                  Pedobacter sp. MW01-1-1|    0.009170|
|10766|                       Spinacia oleracea|    0.007887|
|4830|                    Heyndrickxia oleronia|    0.007867|
|2847|                     Clostridium kluyveri|    0.007701|
|4371|                  Gilliamella sp. ESL0443|    0.007326|
|7330|               Nocardioides campestrisoli|    0.007214|

## 3.3 External validation on independent arthritis cohort

Compute features for arthritis samples using the reference model from the full osteoporosis dataset:

In [ ]:
# Require *files_arth_filtered.txt* with paths to files; set permissions to 755 (rwxr-xr-x)
sbatch arth_calc_features.sbatch

Run the external validation script

In [ ]:
python external_validation_arthritis.py

## 3.4 Cross‑validation on full dataset

In [ ]:
sbatch cv_full.sbatch